In [ ]:
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

### Читаем данные

In [ ]:
# читаем доступные категории
with open('data/categories.txt', 'r') as f:
    categories = f.readlines()
for i in range(len(categories)):
    categories[i] = categories[i].replace('\n', '')
print(categories)

In [ ]:
# читаем данные для обучения
train_data = pd.read_csv('data/train.csv')
train_data.head(5)

### Разметка данных

In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

# загружаем модель и токенизатор
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# категории товаров с описанием
description = {
    'бытовая техника': 'холодильники, стиральные машины, плиты, микроволновки, чайники, пылесосы и другая техника для дома.',
    'обувь': 'кроссовки, ботинки, туфли, сандалии, сапоги и другая обувь для взрослых и детей.',
    'одежда': 'платья, брюки, юбки, футболки, свитеры, куртки, бельё, бюстгальтеры, верхняя одежда.',
    'посуда': 'только тарелки, кружки, чашки для еды, столовые приборы и кухонная утварь; НЕ включает одежду, бельё или чашки бюстгальтеров.',
    'текстиль': 'только постельное бельё, покрывала, наволочки, кухонные полотенца, пледы, портьеры, шторы, ковры и так далее.',
    'товары для детей': 'игрушки, детская одежда, детская мебель, товары для ухода за детьми, коляски, автокресла.',
    'украшения и аксессуары': 'серьги, кольца, браслеты, ожерелья, ремни, шарфы, сумки, очки и другие модные аксессуары.',
    'электроника': 'смартфоны, планшеты, ноутбуки, камеры, наушники, колонки, смарт-часы и прочая электроника.',
    'нет товара': 'отзыв не содержит упоминания конкретного товара.'
}

# категории товаров с примерами
examples = {
    'бытовая техника': 'Заказала стиральную машину, пришла быстро, упаковка целая, но при включении гремит и сильно прыгает по полу.',
    'обувь': 'Купила новые кроссовки, очень удобные.',
    'одежда': 'Чашка бюстгальтера слишком мала, размер не подошёл.',
    'посуда': 'Купила новые кружки для чая, качество отличное.',
    'текстиль': 'Купила постельное бельё из хлопка, качество хорошее.',
    'товары для детей': 'Игрушка пришла быстро, без запаха, ребёнок в восторге.',
    'украшения и аксессуары': 'у очков в белой оправе было деформировано стекло , все как будто "плывет"',
    'электроника': 'Наушники ужасные, звук глухой и один перестал работать через неделю.',
    'нет товара': 'Доставка пришла вовремя, но сервис ужасный.'
}

# дополнительная информации для разметки данных
rules = [
    'Выбирай строго одну из доступных категорий.',
    'Если отзыв не содержит упоминания товара — «нет товара».',
    'Слово «чашка» проверяй по контексту: если речь о бюстгальтере/одежде - «одежда», если речь о посуде - «посуда».',
    'Не добавляй новые категории и не исправляй написание слов.',
    f'Отвечай строго в формате JSON: {{"category": "<название категории>"}}'
]

In [ ]:
# контсруктор промпта для llm
def prompt_builder(review):
    prompt = 'Ты — помощник для классификации текстов. Твоя задача: по тексту отзыва определить, к какой из категорий он относится.\n\n'

    prompt_categories = 'Доступуные категории:\n'
    for cat, desc in description.items():
        prompt_categories += cat + ' - ' + desc + '\n'
    prompt += prompt_categories + '\n'

    prompt_rules = 'Жесткие правила:\n'
    for rule in rules:
        prompt_rules += '- ' + rule + '\n'
    prompt += prompt_rules + '\n'

    prompt_examples = 'Примеры:\n'
    for cat, example in examples.items():
        prompt_examples += 'Отзыв: ' + example + f' Ответ: {{"category": {cat}}}\n'
    prompt += prompt_examples

    return prompt

# получение ответа от llm
def get_llm_answer(prompt, max_tokens=512, temperature=0.0, top_p=1.0, do_sample=False):
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    response = tokenizer.decode(output_ids, skip_special_tokens=True)

    return response

In [ ]:
# разметка тренировочных данных
result = {'text': [], 'category': []}
for text in tqdm(train_data['text'].values, desc='Разметка данных', total=len(train_data)):
    result['text'].append(text)
    answer = get_llm_answer(prompt_builder(text))
    flag = True
    for cat in categories:
        if cat in answer:
            result['category'].append(cat)
            flag = False
            break
    if flag:
        result['category'].append('нет категории')
        print(f'Ошибка при парсинге: {answer}')

### Сохранение результата

In [ ]:
marked_data = pd.DataFrame(result)
marked_data.to_csv('data/marked_data.csv', index=False)